# 📑 Automated Reviewer Assignment Optimization Flow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/visvesh-sathe/reviewer_assignment/blob/main/reviewer_assignment/reviewer_assignment_colab.ipynb)

This standalone Colab workflow solves the **Reviewer Assignment Optimization Problem** using Mixed-Integer Linear Programming (MILP):
- Maximizes paper-reviewer expertise matching scores based on technical areas.
- Enforces strict paper coverage constraints (`reviews_per_paper`) and balanced reviewer workloads (`max_extra_reviews`).
- Enforces Conflict of Interest (COI) exclusions (using the verified `cfg_coi.csv` matrix from `coi_generator`).
- Satisfies mandatory `must_review` reviewer assignments.
- Fairly appoints balanced **Champions** and **Co-Champions** for every paper, excluding Committee Chairs.
- Produces comprehensive summary reports, distribution histograms, and the final color-coded deliverable spreadsheet (`final_reviewer_assignment_sheet.xls`).

---
### 📋 4-Step Interactive Workflow:
1. **Step 1:** Setup workspace and install PuLP & Excel optimization dependencies.
2. **Step 2:** Download reference templates or upload your customized input files (`cfg_paper_area.csv`, `cfg_reviewer_expertise.csv`, `cfg_coi.csv`, `cfg_must_review.csv`, `cfg_config.yaml`).
3. **Step 3:** Configure workload parameters (`reviews_per_paper`, `max_extra_reviews`) and run the optimization engine.
4. **Step 4:** Inspect workload distribution charts inline and download the complete deliverables package.

---
## 🛠️ Step 1: Install Dependencies & Setup Workspace
Installs `pulp` MILP solver and optimization libraries, sets up directory structures, and clones the repository if running on Google Colab.

In [ ]:
#@title Step 1: Install Dependencies & Setup Workspace { display-mode: "form" }
import os
import sys
import shutil
from pathlib import Path

# 1. Install dependencies (PuLP 3.x with bundled CBC solver, openpyxl, pyyaml)
print("🔧 Installing optimization and spreadsheet packages...")
!pip install -q "pulp>=2.8,<4.0" openpyxl pandas pyyaml matplotlib

# 2. Clone or sync repository in Colab
repo_dir = Path("/content/reviewer_assignment")
if not repo_dir.exists():
    print("\n📡 Fetching latest reviewer_assignment codebase from GitHub...")
    !git clone https://github.com/visvesh-sathe/reviewer_assignment.git /content/reviewer_assignment
else:
    print("\n🔄 Syncing to latest commit from GitHub...")
    !cd /content/reviewer_assignment && git fetch origin main && git reset --hard origin/main

print("\n📌 Active Git Commit in Colab:")
!cd /content/reviewer_assignment && git log -n 1 --oneline

# 3. Enter workspace directory
workspace = repo_dir / "reviewer_assignment"
if workspace.exists():
    os.chdir(str(workspace))
    print(f"\n✅ Environment Ready! Current directory: {os.getcwd()}")
    print("\n📁 Files available in input_files/:")
    for f in sorted(os.listdir("input_files")):
        print(f"   • {f}")
    print("\n📁 Files available in reference_templates/:")
    for f in sorted(os.listdir("reference_templates")):
        print(f"   • {f}")
else:
    print("⚠️ Could not locate workspace directory. Please re-run this cell.")


---
## 📥 Step 2: Download Templates & Upload Input Files
In this panel, you can:
1. **Download Input Templates First (Part A):** Download the blank/reference template files (`reviewer_expertise`, `paper_area`, `must_review`, `config.yaml`, and the deliverable sheet template) to see the expected columns and formatting.
   > 💡 *Note on COI:* You do **not** need to manually fill out a COI template! Your Conflict of Interest matrix (`cfg_coi.csv`) is automatically produced by the [`coi_generator`](https://colab.research.google.com/github/visvesh-sathe/reviewer_assignment/blob/main/coi_generator/coi_generator.ipynb) workflow.
2. **Upload Your Completed Files (Part B):** Upload your completed data files and the `cfg_coi.csv` generated by `coi_generator`.

*(If you do not upload a custom file, the pre-loaded default sample file is used automatically for testing.)*

In [ ]:
#@title Step 2: Reference Templates & Input Files Panel { display-mode: "form" }
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import files

# Auto-recover working directory if needed
workspace = Path("/content/reviewer_assignment/reviewer_assignment")
if not os.path.exists("input_files") and workspace.exists():
    os.chdir(str(workspace))

if not os.path.exists("input_files") or not os.path.exists("reference_templates"):
    print("⚠️ Workspace files not found. Please click the Play button on Step 1 first!")
else:
    # ----------------- SECTION A: DOWNLOAD REFERENCE TEMPLATES -----------------
    ref_dropdown = widgets.Dropdown(
        options=[
            ("📦 All Input Templates (ZIP Package - excludes COI)", "reference_templates/reference_templates.zip"),
            ("1. Reviewer Expertise Template (cfg_reviewer_expertise.csv)", "reference_templates/reference_reviewer_expertise.csv"),
            ("2. Paper Technical Areas Template (cfg_paper_area.csv)", "reference_templates/reference_paper_area.csv"),
            ("3. Must-Review Constraints Template (cfg_must_review.csv)", "reference_templates/reference_must_review.csv"),
            ("4. Master Configuration YAML (cfg_config.yaml)", "reference_templates/reference_config.yaml"),
            ("5. Deliverable Spreadsheet Template (reference_assignment_sheet_template.xls)", "reference_templates/reference_assignment_sheet_template.xls"),
            ("6. Sample COI Matrix for offline test runs (reference_coi.csv)", "reference_templates/reference_coi.csv"),
        ],
        description="Template:",
        layout=widgets.Layout(width="480px")
    )

    btn_download_ref = widgets.Button(
        description="⬇️ Download Selected File",
        button_style="info",
        icon="download",
        layout=widgets.Layout(width="240px")
    )

    btn_download_all = widgets.Button(
        description="📦 Download All Templates (.ZIP)",
        button_style="primary",
        icon="archive",
        layout=widgets.Layout(width="260px")
    )

    def on_download_ref_clicked(b):
        path = ref_dropdown.value
        if os.path.exists(path):
            print(f"⬇️ Downloading '{os.path.basename(path)}' to your browser...")
            files.download(path)
        else:
            print(f"Error: '{path}' not found.")

    def on_download_all_clicked(b):
        zip_path = "reference_templates/reference_templates.zip"
        if os.path.exists(zip_path):
            print(f"⬇️ Downloading '{zip_path}' to your browser...")
            files.download(zip_path)
        else:
            print(f"Error: '{zip_path}' not found.")

    btn_download_ref.on_click(on_download_ref_clicked)
    btn_download_all.on_click(on_download_all_clicked)

    # ----------------- SECTION B: UPLOAD CUSTOM FILES -----------------
    btn_areas = widgets.FileUpload(description="1. Tech Areas", accept=".csv", multiple=False, layout=widgets.Layout(width="200px"))
    btn_expertise = widgets.FileUpload(description="2. Expertise", accept=".csv", multiple=False, layout=widgets.Layout(width="200px"))
    btn_coi = widgets.FileUpload(description="3. Upload COI Matrix", accept=".csv", multiple=False, layout=widgets.Layout(width="200px"))
    btn_must = widgets.FileUpload(description="4. Must-Review", accept=".csv", multiple=False, layout=widgets.Layout(width="200px"))
    btn_config = widgets.FileUpload(description="5. Config (YAML)", accept=".yaml,.yml", multiple=False, layout=widgets.Layout(width="200px"))

    btn_apply = widgets.Button(description="💾 Save & Stage Uploaded Files", button_style="success", icon="check", layout=widgets.Layout(width="280px", height="38px"))
    output_status = widgets.Output()

    def on_apply_clicked(b):
        with output_status:
            output_status.clear_output()
            mapping = [
                (btn_areas, "input_files/cfg_paper_area.csv", "Paper Technical Areas"),
                (btn_expertise, "input_files/cfg_reviewer_expertise.csv", "Reviewer Expertise"),
                (btn_coi, "input_files/cfg_coi.csv", "COI Matrix (from coi_generator)"),
                (btn_must, "input_files/cfg_must_review.csv", "Must-Review Constraints"),
                (btn_config, "input_files/cfg_config.yaml", "Configuration YAML"),
            ]
            saved_count = 0
            for uploader, dest_path, label in mapping:
                if uploader.value:
                    file_info = uploader.value[0] if isinstance(uploader.value, list) else list(uploader.value.values())[0]
                    content = file_info["content"]
                    with open(dest_path, "wb") as f:
                        f.write(content)
                    print(f"✅ Successfully staged custom {label} -> {dest_path}")
                    saved_count += 1
            if saved_count == 0:
                print("ℹ️ No new files selected. Existing default files in 'input_files/' will be used.")
            else:
                print(f"\n🎉 Successfully staged {saved_count} file(s)! Ready for Step 3.")

    btn_apply.on_click(on_apply_clicked)

    display(HTML("<h4><b>Part A: Download Blank Input Templates (excluding COI)</b></h4>"))
    display(HTML("<p style='color:#555;'>Download the templates to format your papers, reviewers, constraints, and configuration. (The COI matrix is generated by <code>coi_generator</code>).</p>"))
    display(widgets.HBox([btn_download_all]))
    display(HTML("<p style='margin-top:8px; margin-bottom:4px;'><i>Or download individual template files:</i></p>"))
    display(widgets.HBox([ref_dropdown, btn_download_ref]))
    display(HTML("<br><h4><b>Part B: Upload Your Customized Files & Generated COI Matrix</b></h4>"))
    display(widgets.VBox([
        widgets.HBox([btn_areas, btn_expertise]),
        widgets.HBox([btn_coi, btn_must]),
        widgets.HBox([btn_config]),
        widgets.Box([btn_apply], layout=widgets.Layout(margin="15px 0 0 0")),
        output_status
    ]))


---
## 🚀 Step 3: Run Reviewer Allocation (MILP Optimizer)
Runs the Mixed-Integer Linear Programming optimization to assign reviewers, appoint balanced Champions and Co-Champions, and populate the deliverable spreadsheet (`final_reviewer_assignment_sheet.xls`).

You can adjust `reviews_per_paper` and `max_extra_reviews` using the form controls below.

In [ ]:
#@title Step 3: Execute Reviewer Allocation Flow { display-mode: "form" }

#@markdown ### ⚙️ Reviewer Workload Settings:
#@markdown Number of reviews required per paper (default loaded from cfg_config.yaml):
reviews_per_paper = 8 #@param {type:"integer"}

#@markdown Max extra reviews per reviewer above fair minimum baseline (default: 2):
max_extra_reviews = 2 #@param {type:"integer"}

import os
import yaml

# Auto-recover working directory if needed
workspace = Path("/content/reviewer_assignment/reviewer_assignment")
if not os.path.exists("input_files") and workspace.exists():
    os.chdir(str(workspace))

# Read defaults from cfg_config.yaml if available
config_file = "input_files/cfg_config.yaml"
yaml_reviews_per_paper = 8
yaml_max_extra = 2
if os.path.exists(config_file):
    try:
        with open(config_file, "r") as f:
            cfg_data = yaml.safe_load(f) or {}
            yaml_reviews_per_paper = int(cfg_data.get("reviews_per_paper", cfg_data.get("min_reviews_per_paper", 8)))
            yaml_max_extra = int(cfg_data.get("max_extra_reviews", 2))
    except Exception as e:
        pass

# Fallback to YAML default if non-positive value entered
active_reviews_per_paper = reviews_per_paper if reviews_per_paper > 0 else yaml_reviews_per_paper
active_max_extra = max_extra_reviews if max_extra_reviews >= 0 else yaml_max_extra

print(f"🚀 Running Reviewer Allocation Optimizer:")
print(f"   • reviews_per_paper: {active_reviews_per_paper} (YAML default: {yaml_reviews_per_paper})")
print(f"   • max_extra_reviews: {active_max_extra} (YAML default: {yaml_max_extra})\n")

!python3 reviewer_allocation.py --reviews-per-paper {active_reviews_per_paper} --max-extra-reviews {active_max_extra}


---
## 📊 Step 4: View Visualizations & Download Deliverables
Displays all workload distribution charts inline and downloads the final `reviewer_assignment_results.zip` package containing all assignment matrices, vectors, summary reports, and the filled deliverable spreadsheet (`final_reviewer_assignment_sheet.xls`).

In [ ]:
#@title Step 4: Display Charts & Download Deliverables Package { display-mode: "form" }
import glob
import os
import shutil
from IPython.display import Image, display
from google.colab import files

# Auto-recover working directory if needed
workspace = Path("/content/reviewer_assignment/reviewer_assignment")
if not os.path.exists("output_files") and workspace.exists():
    os.chdir(str(workspace))

# 1. Display Inline Charts
png_files = sorted(glob.glob("output_files/*.png"))
if png_files:
    print("📊 Workload Distribution Charts:")
    for png in png_files:
        print(f"\n--- {os.path.basename(png)} ---")
        display(Image(filename=png, width=800))
else:
    print("No charts found. Please run Step 3 first.")

# 2. Summary Report Preview
report_path = "output_files/summary_report.txt"
if os.path.exists(report_path):
    print("\n" + "="*60)
    print("📄 SUMMARY REPORT PREVIEW:")
    print("="*60)
    with open(report_path, "r") as f:
        lines = f.readlines()
        print("".join(lines[:45]))
        if len(lines) > 45:
            print(f"... ({len(lines) - 45} more lines in output_files/summary_report.txt)")

# 3. Package Deliverables
output_dir = Path("output_files")
archive_base = "reviewer_assignment_results"
zip_path = f"{archive_base}.zip"

if output_dir.exists():
    shutil.make_archive(archive_base, "zip", str(output_dir))
    print(f"\n✅ Successfully packaged all assignment outputs into '{zip_path}'.")
    
    # Trigger downloads
    excel_deliverable = "output_files/final_reviewer_assignment_sheet.xls"
    if os.path.exists(excel_deliverable):
        print(f"⬇️ Downloading final assignment spreadsheet ({excel_deliverable})...")
        files.download(excel_deliverable)
    
    print(f"⬇️ Downloading complete deliverables package ({zip_path})...")
    files.download(zip_path)
else:
    print("❌ 'output_files/' directory not found. Please run Step 3 first.")
